#### Imports:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.context import SparkContext
from pyspark import SparkConf
from pyspark.sql.functions import (
    udf, col, when, asc, desc, lit, coalesce,
    mean, sum, avg, rand, stddev,
    count, countDistinct,
    format_number, isnan,
    asc, desc, mean, 
    rank, lag, lead,
)
from pyspark.sql.window import Window

from pyspark.sql.types import (
    StructField, StructType, LongType, TimestampType,
    StringType, IntegerType, 
    FloatType, BooleanType,
    DateType,
)

In [ ]:
import pyspark, datetime, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from sparking import *
# from sparking import employees_df, bonus_df

#### Creating the Spark Entry point (`SparkSession` / `SparkContext`)

In [ ]:
my_conf = SparkConf().setAppName("My_Spark_App") \
    .set("spark.sql.shuffle.partitions", "2") \
    .set("spark.driver.memory", "4g") \
    .set("spark.executor.memory", "2g") \
    .set("spark.sql.autoBroadcastJoinThreshold", 10_000_000) \
    .set("spar.sql.adaptive.enabled", False)

spark = SparkSession.builder \
    .appName("RenamedSparkApp") \
    .config(conf=my_conf) \
    .getOrCreate()

sc = spark.sparkContext
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Enable eager evaluation for better formatting of the output
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
# Disable Broadcast Join
spark.conf.set("spar.sql.autoBroadcastJoinThreshold", -1)

In [ ]:
# spark.sparkContext.getConf().getAll()
# spark.conf.get("spark.sql.parquet.filterPushDown")
# spark.conf.get("spark.sql.sources.bucketing.enabled")
# spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
# spark.conf.get("spark.sql.warehouse.dir")

#### How to connect MySQL in Apache Spark?


-  **Download the MySQL JDBC Driver**:

    -   You can download the `MySQL Connector/J` (JDBC driver) from the official MySQL website: MySQL Connector/J.
    -   Choose the version that matches your environment (e.g., mysql-connector-java-8.0.30.jar).


- **How to nsure the Driver is Available to Spark**: 
    -    Place the downloaded `mysql-connector-java-8.0.xx.jar` file in a directory accessible by your Spark environment. Note the full path to this JAR file.

    -   **Placing the JAR in Spark’s jars Directory**: You can also place the MySQL JDBC JAR file in Spark’s `jars` folder, typically located in your Spark installation directory (`$SPARK_HOME/jars`). Spark will automatically load any JARs found in this folder when it starts.

    - Specify the JAR file for Spark Submit using **spark-submit** Command:

        - Use the `--jars` option to include the MySQL JDBC driver when you submit your Spark job.
        - `spark-shell --jars /Users/am/mydocs/Software_Development/Big_Data/elements_of_spark/mysql-connector-j-8.0.32.jar scripts/read_mysql.py`

    - Specify the JAR file for Spark Shell using **spark-shell** Command:
        - `spark-shell --jars /Users/am/mydocs/Software_Development/Big_Data/elements_of_spark/mysql-connector-j-8.0.32.jar`
        - `pyspark --jars /Users/am/mydocs/Software_Development/Big_Data/elements_of_spark/mysql-connector-j-8.0.32.jar`

In [ ]:
# JDBC URL format: jdbc:mysql://<host>:<port>/<db-name>
# jdbc_url = "jdbc:mysql://localhost:3306/interview_questions"
jdbc_url = "jdbc:mysql://localhost:3306"
mysql_driver = f"{os.environ['SPARK_HOME']}/jars/mysql-connector-j-8.0.32.jar"

# Connection properties
connection_properties = {
    "user": os.environ["MYSQL_USERNAME"],
    "password": os.environ["MYSQL_PASSWORD"],
    "driver": "com.mysql.cj.jdbc.Driver",
}

In [ ]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Data Ingestion from MySQL into  Spark") \
    .config("spark.jars", mysql_driver) \
    .getOrCreate()

##### <b style="color:magenta">Read from RDBMS</b>

In [ ]:
print({**connection_properties, "dbtable": "Employee", "url": jdbc_url + "/IQ"})

In [ ]:
# Reading table from MySQL
# df = spark.read.jdbc(url=jdbc_url+'/interview_questions', table="Bonus", properties=connection_properties)


options = {
    **connection_properties,
    "dbtable": "Employee",
    "url": jdbc_url + "/IQ",
}
df = spark.read.format("jdbc").options(**options).load()

df.show()


In [ ]:
df.write.mode("overwrite").option("header", "true").csv(
    "/Users/am/mydocs/Software_Development/noteshub/data/EMP.csv"
)


##### <b style="color:magenta">Write to RDBMS</b>

In [ ]:
## Write OPTION 1
# emp_df.write.jdbc(url=jdbc_url+'/IQ', table="Employees", mode="overwrite", properties=connection_properties)

In [ ]:
# SQL statement with additional constraints
create_emp_sql = """CREATE TABLE employee (
	EMPLOYEE_ID INT NOT NULL PRIMARY KEY AUTO_INCREMENT,
	FIRST_NAME CHAR(25),
	LAST_NAME CHAR(25),
	DEPARTMENT CHAR(25),
	SALARY INT(15),
	JOINING_DATE CHAR(25),
	EMAIL CHAR(100),
    DOB TIMESTAMP,
);
"""

# SQL statement with additional constraints
create_bns_sql = """CREATE TABLE Bonus IF NOT EXISTS (
	EMPLOYEE_REF_ID INT,
	BONUS_AMOUNT INT(10),
	BONUS_DATE DATETIME,
	FOREIGN KEY (EMPLOYEE_REF_ID) REFERENCES Employee(EMPLOYEE_ID) ON DELETE CASCADE
);
"""

In [ ]:
# JDBC options with the preactions to create the table with constraints
jdbc_options = {
    "url": jdbc_url+'/IQ',
    "dbtable": "Employee",
    "user": connection_properties["user"],
    "password": connection_properties["password"],
    "driver": connection_properties["driver"],
}

In [ ]:
emp_pddf = employees_df(30)

# emp_df = spark.createDataFrame(emp_pddf)
# emp_df = emp_df.fillna({'salary': 0,})
# emp_df.na.fill({"EMAIL": "UNKNOWN"})
# emp_df = emp_df.replace({None: "UNKNOWN"}, subset=['EMAIL'])

In [ ]:
emp_df = spark.createDataFrame(emp_pddf)

In [ ]:
# emp_df = emp_df.select('FIRST_NAME','LAST_NAME','DEPARTMENT','SALARY','JOINING_DATE','EMAIL')
# emp_df.show()

In [ ]:
# Update JDBC options
jdbc_options["preactions"] = create_emp_sql # This will create the table if it doesn't exist with the primary key constraint

In [ ]:
# emp_df.write.jdbc(
#     url=jdbc_url + "/IQ",
#     table="Employee",
#     mode="overwrite",
#     properties=connection_properties,
# )


`NOTES`: To append into an existing table, data must be validated in accordance with table constraints.

In [ ]:
## Write OPTION 2
# To append into an existing table, data must be validated in accordance with table constraints.
emp_df.write \
    .format("jdbc") \
    .options(**jdbc_options) \
    .mode("overwrite") \
    .save()


In [ ]:
# emp_df.toPandas()

#### Create a View And Run Queries

In [ ]:
emp_df.createOrReplaceTempView("emp_df_view")

In [ ]:
spark.sql("SELECT * FROM emp_df_view LIMIT 2").show()

```sql

SELECT department, COUNT(*) AS num_employees, AVG(salary) AS avg_salary
FROM Employee
-- WHERE JOINING_DATE BETWEEN '2017-01-01' AND '2017-06-31'
WHERE department IN ('HR', 'IT', 'Finance')
GROUP BY department
-- HAVING AVG(salary) > 30000
HAVING avg_salary > 30000
ORDER BY avg_salary DESC
LIMIT 5;
```

In [ ]:
# Initialize Spark session
spark = SparkSession.builder.appName("EmployeeAnalysis").getOrCreate()

# # Reading table from MySQL
emp_df = spark.read.jdbc(
    url=jdbc_url + "/interview_questions",
    # url=jdbc_url + "/IQ",
    table="Employees",
    properties=connection_properties,
)

In [ ]:
emp_df

In [ ]:
# Filter departments
filtered_emp_df = emp_df.filter(col("department").isin("HR", "IT", "Finance"))

# Group by department and aggregate
grouped_emp_df = filtered_emp_df.groupBy("department").agg(
    count("*").alias("num_employees"), avg("salary").alias("avg_salary")
)

# Filter on avg_salary (equivalent to HAVING clause)
having_emp_df = grouped_emp_df.filter(col("avg_salary") > 30000)

# Order and limit
result_emp_df = having_emp_df.orderBy(col("avg_salary").desc()).limit(5)

# Show results
result_emp_df.show()


# `Stop` SparkSession

In [ ]:
# Stop Spark Session
spark.stop()